In [7]:
import os
import numpy as np
import plotly.graph_objects as go

# =============================================================================
# 1) Configuration
# =============================================================================
np.random.seed(42)
output_dir = './data'  # Changed to local directory
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# 2) Synthetic IoT Data Generation (~35 days of hourly readings)
# =============================================================================
num_samples = 20_000
t = np.linspace(0, 24 * 35, num_samples)

# Latent factors: daily cycle (sin, cos), weekly cycle (sin, cos), linear drift
daily   = np.stack([np.sin(2 * np.pi * t / 24),
                    np.cos(2 * np.pi * t / 24)], axis=1)
weekly  = np.stack([np.sin(2 * np.pi * t / (24*7)),
                    np.cos(2 * np.pi * t / (24*7))], axis=1)
drift   = (t / (24*35))[:, None]

# True top-layer latent (5 dims)
h_true = np.concatenate([daily, weekly, drift], axis=1)

# Hierarchy dims: 6 → 16 → 12 → 8 → 5
dims = [5, 16, 12, 8, h_true.shape[1]]

# Random projection weights (smaller scale to avoid overflow)
W = [np.random.randn(dims[i], dims[i+1]) * 0.01
     for i in range(len(dims) - 1)]

# Generate x_data by descending through the projections
h = h_true.copy()
for W_mat in reversed(W):
    h = h @ W_mat.T \
        + 0.05 * np.random.randn(num_samples, W_mat.shape[0])
x_data = h

# Inject a block of anomalies
anom_idx = np.arange(15_000, 15_100)
x_data[anom_idx] += np.random.randn(len(anom_idx), x_data.shape[1]) * 4

# =============================================================================
# 3) Define Deep Predictive Coding Network
# =============================================================================
class PCLayer:
    def __init__(self, in_d, out_d):
        self.W = np.random.randn(in_d, out_d) * 0.01

    def inference(self, x, iters=25, lr_h=0.01):
        """
        Iterative inference: prediction <-> error loop
        """
        h = np.zeros((x.shape[0], self.W.shape[1]))
        for _ in range(iters):
            x_hat = h @ self.W.T
            err   = x - x_hat
            h    += lr_h * (err @ self.W)
        return h, err

    def update(self, x, h, lr=1e-4):
        """
        Weight update via simple Hebbian-like rule
        """
        grad = (x.T @ h) / x.shape[0]
        self.W += lr * grad

class DeepPCNet:
    def __init__(self, dims):
        self.layers = [PCLayer(dims[i], dims[i+1])
                       for i in range(len(dims)-1)]

    def train(self, data, epochs=60):
        """
        Train by alternating upward inference and top-down weight updates
        """
        for _ in range(epochs):
            latents = [data]
            # upward pass
            for layer in self.layers:
                h, _ = layer.inference(latents[-1])
                latents.append(h)
            # top-down update
            for i, layer in enumerate(self.layers):
                layer.update(latents[i], latents[i+1])

    def reconstruct(self, data):
        """
        Full reconstruct: upward inference, then downward projection
        """
        h = data
        for layer in self.layers:
            h, _ = layer.inference(h)
        for layer in reversed(self.layers):
            h = h @ layer.W.T
        return h

# =============================================================================
# 4) Train network and reconstruct
# =============================================================================
pc_net = DeepPCNet(dims)
pc_net.train(x_data, epochs=60)
recon  = pc_net.reconstruct(x_data)

# Compute per-sample reconstruction error
errors = np.linalg.norm(x_data - recon, axis=1)

# =============================================================================
# 5) Plotly Visualizations
# =============================================================================

# -- 5.1) Channel 0: Original vs Reconstructed
idx = np.arange(1, 20_000)
fig1 = go.Figure([
    go.Scatter(x=idx, y=x_data[idx, 0], mode='lines', name='Original'),
    go.Scatter(x=idx, y=recon[idx, 0], mode='lines', name='Reconstructed'),
])
fig1.update_layout(
    title="Sensor Channel 0: Original vs Reconstructed",
    xaxis_title="Sample Index",
    yaxis_title="Reading"
)
fig1.write_html(os.path.join(output_dir, 'pc_channel0.html'))

# -- 5.2) Reconstruction Error Over Time with Anomaly Highlight
fig2 = go.Figure([
    go.Scatter(x=np.arange(num_samples), y=errors,
               mode='lines', name='Error Norm'),
])
# Calculate threshold for anomaly detection (mean + 3*std)
threshold = np.mean(errors) + 3 * np.std(errors)

# Add threshold line
fig2.add_hline(y=threshold, line_dash="dash", line_color="red",
               annotation_text="Anomaly Threshold", 
               annotation_position="top right")

# Find and highlight anomaly regions
anomaly_indices = np.where(errors > threshold)[0]
if len(anomaly_indices) > 0:
    # Group consecutive anomaly indices
    anomaly_groups = np.split(anomaly_indices, np.where(np.diff(anomaly_indices) != 1)[0] + 1)
    
    # Add vrect for each anomaly group
    for group in anomaly_groups:
        if len(group) > 0:
            fig2.add_vrect(x0=group[0], x1=group[-1],
                          fillcolor="LightSalmon", opacity=0.3,
                          line_width=0, annotation_text="Anomaly",
                          annotation_position="top left")
fig2.update_layout(
    title="Reconstruction Error Over Time",
    xaxis_title="Sample Index",
    yaxis_title="Error Norm"
)
fig2.write_html(os.path.join(output_dir, 'pc_error.html'))

# -- 5.3) True vs Estimated Top Latent Dimension 0
# Compute estimated top latent
h_est = x_data.copy()
for layer in pc_net.layers:
    h_est, _ = layer.inference(h_est)

fig3 = go.Figure([
    go.Scatter(
        x=h_true[:, 0], y=h_est[:, 0],
        mode='markers', marker=dict(size=3, opacity=0.4)
    )
])
fig3.update_layout(
    title="True vs Estimated Top Latent Dimension 0",
    xaxis_title="True Top Latent",
    yaxis_title="Estimated Top Latent"
)
fig3.write_html(os.path.join(output_dir, 'pc_latent.html'))

print("HTML visualizations saved to:", output_dir)


HTML visualizations saved to: ./data
